# 2_linelist_modify

## Create the linelist
- Convert the linelist we want to use to ispec-readable form
- Crossmatch our linelist with input/linelists/transitions/VALD.300_1100nm/atomic_lines.tsv

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import logging
import multiprocessing
from multiprocessing import Pool
import matplotlib.pyplot as plt
from scipy.stats import norm

from inlist import target, PIPELINE_OPTIONS_OVERRIDE
from run_context import apply_context_globals

ctx = apply_context_globals(
    globals(),
    target=target,
    pipeline_options_override=PIPELINE_OPTIONS_OVERRIDE,
    prepare_io=True,
)

## Part I: Convert the linelist

In [ ]:
# ELEMENT_MAP, including 1-70
ELEMENT_MAP = {
    1: "H", 2: "He", 3: "Li", 4: "Be", 5: "B",
    6: "C", 7: "N", 8: "O", 9: "F", 10: "Ne",
    11: "Na", 12: "Mg", 13: "Al", 14: "Si", 15: "P",
    16: "S", 17: "Cl", 18: "Ar", 19: "K", 20: "Ca",
    21: "Sc", 22: "Ti", 23: "V", 24: "Cr", 25: "Mn",
    26: "Fe", 27: "Co", 28: "Ni", 29: "Cu", 30: "Zn",
    31: "Ga", 32: "Ge", 33: "As", 34: "Se", 35: "Br",
    36: "Kr", 37: "Rb", 38: "Sr", 39: "Y", 40: "Zr",
    41: "Nb", 42: "Mo", 43: "Tc", 44: "Ru", 45: "Rh",
    46: "Pd", 47: "Ag", 48: "Cd", 49: "In", 50: "Sn",
    51: "Sb", 52: "Te", 53: "I", 54: "Xe", 55: "Cs",
    56: "Ba", 57: "La", 58: "Ce", 59: "Pr", 60: "Nd",
    61: "Pm", 62: "Sm", 63: "Eu", 64: "Gd", 65: "Tb",
    66: "Dy", 67: "Ho", 68: "Er", 69: "Tm", 70: "Yb"
}


In [ ]:
# Create a new linelist.
def convert_melendez_to_ispec(input_filepath, output_filepath):
    """
    Convert our own linelist (Melendez 2014) to a iSpec readable linelist file format.

    Parameters:
    - input_filepath: str, Melendez linelist file path
    - output_filepath: str, the output iSpec linelist file path
    """
    total_lines = 0
    valid_lines = 0
    skipped_lines = 0

    with open(input_filepath, 'r') as infile, open(output_filepath, 'w') as outfile:
        # write the header
        header = ("element\twave_A\twave_nm\tloggf\tlower_state_eV\tlower_state_cm1\tlower_j\t"
                  "upper_state_eV\tupper_state_cm1\tupper_j\tupper_g\tlande_lower\tlande_upper\t"
                  "spectrum_transition_type\trad\tstark\twaals\twaals_single_gamma_format\t"
                  "turbospectrum_fdamp\tspectrum_fudge_factor\ttheoretical_depth\ttheoretical_ew\t"
                  "lower_orbital_type\tupper_orbital_type\tmolecule\tspectrum_synthe_isotope\t"
                  "ion\tspectrum_moog_species\tturbospectrum_species\twidth_species\treference_code\t"
                  "spectrum_support\tturbospectrum_support\tmoog_support\twidth_support\tsynthe_support\tsme_support\n")
        outfile.write(header)

        for line_num, line in enumerate(infile, start=1):
            total_lines += 1
            data = line.strip().split()

            # skip the invalid rows
            if len(data) < 5 or not data[0].replace('.', '', 1).isdigit():
                skipped_lines += 1
                continue

            try:
                wave_A = float(data[0])
                ion_value = float(data[1])
                lower_state_eV = float(data[2])
                loggf = float(data[3])
                waals = float(data[4])

                # Obtain atomic number and ionization stage.
                atomic_number = int(ion_value)
                ion_stage = int((ion_value - atomic_number) * 10) + 1   # iSpec uses 1 for neutral atoms.

                # Obtain the element symbol.
                element = ELEMENT_MAP.get(atomic_number, "Unknown")
                if element == "Unknown":
                    skipped_lines += 1
                    continue

                # Build the element column.
                element_str = f"{element} {ion_stage}"

                # Calculate the remaining fields.
                wave_nm = wave_A / 10.0
                lower_state_cm1 = lower_state_eV * 8065.54

                # Build the species column.
                moog_species = f"{ion_value:.1f}"
                turbospectrum_species = f"{atomic_number:.6f}"
                width_species = f"{atomic_number}.0{ion_stage-1}"

                # Construct the output row.
                line_data = [
                    element_str, wave_A, wave_nm, loggf, lower_state_eV, lower_state_cm1, 0.0,
                    0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                    "GA", 0.0, 0.0, waals, waals, 0.0, 1.0, 0.0, 0.0,
                    "d", "f", "F", 0, ion_stage, moog_species, turbospectrum_species, width_species,
                    "Melendez2014", "T", "T", "T", "T", "T", "T"
                ]

                # Write the row.
                outfile.write("\t".join(map(str, line_data)) + "\n")
                valid_lines += 1

            except ValueError as ve:
                skipped_lines += 1
                print(f"Error parsing line {line_num}: {line.strip()} - {ve}")
                continue

    print(f"Total lines processed: {total_lines}")
    print(f"Valid lines written: {valid_lines}")
    print(f"Lines skipped: {skipped_lines}")
    print(f"The iSpec readable linelist has been saved in '{output_filepath}'")


In [ ]:
# Run the conversion.
convert_melendez_to_ispec(input_linelist_path, converted_linelist_path)

# Read the linelist.
#linelist_path = "linelist_meledez2014_ispec.txt"


## Part II: Crossmatch

Group lines by element and ionization stage. Match wavelength within 0.015 Angstrom and lower excitation potential within the configured 0.02 or 0.01 eV tolerance. Inspect the matches before replacing loggf and excitation potential with the selected Melendez values; preserve the remaining atomic data.

In [ ]:
def split_by_element(vald_df, melendez_df):
    """Group VALD and Melendez DataFrames by element for crossmatching. Return dictionaries mapping each common element to its rows. Both input tables must contain element labels."""
    # Find the common elements.
    elements = set(vald_df['element'].unique()) & set(melendez_df['element'].unique())

    # Initialize the dictionaries.
    vald_dict = {}
    melendez_dict = {}

    # Group by element.
    for elem in sorted(elements):
        vald_elem_df = vald_df[vald_df['element'] == elem].copy()
        mel_elem_df = melendez_df[melendez_df['element'] == elem].copy()

        if not vald_elem_df.empty and not mel_elem_df.empty:
            vald_dict[elem] = vald_elem_df
            melendez_dict[elem] = mel_elem_df

    print(f"Common elements in the two tables: {len(vald_dict)}; symbols: {list(vald_dict.keys())}")
    return vald_dict, melendez_dict

In [ ]:

def cross_match_per_element(vald_df, melendez_df, output_linelist_folder, output_linelist_path,
                            wave_tol=0.015, ep_tol=0.02):
    """Crossmatch each element and combine successful matches. vald_df and melendez_df contain the input lines; output_linelist_folder stores per-element comparisons; output_linelist_path stores the combined table. wave_tol and ep_tol set wavelength and excitation-potential tolerances."""
    vald_dict, mel_dict = split_by_element(vald_df, melendez_df)
    os.makedirs(output_linelist_folder, exist_ok=True)
    all_crossmatched_df = []
    total_melendez_lines = 0

    for element in vald_dict.keys():
        v_df = vald_dict[element].copy()
        m_df = mel_dict.get(element, pd.DataFrame()).copy()
        total_melendez_lines += len(m_df)

        if m_df.empty:
            print(f"[{element}] has no Melendez entries; skipping")
            continue

        matched_rows = []
        updated_rows = []

        for _, mel_row in m_df.iterrows():
            mel_wave = mel_row["wave_A"]
            mel_ep = mel_row["lower_state_eV"]

            candidates = v_df[
                (np.abs(v_df["wave_A"] - mel_wave) <= wave_tol) &
                (np.abs(v_df["lower_state_eV"] - mel_ep) <= ep_tol)
            ]

            if not candidates.empty:
                val_row = candidates.iloc[0]  # Use the first matching VALD row.

                # Record the comparison.
                matched_rows.append({
                    "element": element,
                    "vald_wave_A": val_row["wave_A"],
                    "melendez_wave_A": mel_wave,
                    "vald_lower_state_eV": val_row["lower_state_eV"],
                    "melendez_lower_state_eV": mel_ep
                })

                # Update the requested columns.
                row_copy = val_row.copy()
                for col in ["wave_A", "wave_nm", "loggf", "lower_state_eV", "lower_state_cm1", "reference_code"]:
                    row_copy[col] = mel_row[col]
                updated_rows.append(row_copy)

        print(f"Melendez2014 [{element}] lines: {len(m_df)}; crossmatched: {len(matched_rows)} lines")

        # Save the comparison table.
        comparison_df = pd.DataFrame(matched_rows)
        comparison_path = os.path.join(output_linelist_folder, f"{element}_comparison.csv")
        comparison_df.to_csv(comparison_path, index=False)

        # Save matched and updated rows only.
        if updated_rows:
            updated_df = pd.DataFrame(updated_rows)
            crossmatch_path = os.path.join(output_linelist_folder, f"{element}_crossmatch.tsv")
            updated_df.to_csv(crossmatch_path, sep="\t", index=False)
            all_crossmatched_df.append(updated_df)

    # Combine crossmatched DataFrames across elements.
    final_df = pd.concat(all_crossmatched_df, ignore_index=True)
    final_df.to_csv(output_linelist_path, sep="\t", index=False)

    # Summarize the matches.
    print("\n===== Cross Match Summary =====")
    print(f"Number of elements: {len(vald_dict)}")
    print(f"Original Melendez line count: {total_melendez_lines}")
    print(f"Successfully crossmatched line count: {len(final_df)}")
    print(f"Combined output saved to: {output_linelist_path}")

In [ ]:
vald_df = pd.read_csv(vald_atomic_lines_path, sep="\t")
melendez_df = pd.read_csv(converted_linelist_path, sep="\t")

single_output_folder = output_linelist_folder + "single_element_cross"
final_crossmatch_output = output_linelist_path

cross_match_per_element(vald_df, melendez_df, single_output_folder, final_crossmatch_output, ep_tol=0.01)

# Export an iSpec transition table

The supplied crossmatched linelist is ready for the example. Run this notebook only when intentionally rebuilding it from your licensed VALD table.

In [ ]:
# robust_crossmatch_update_from_melendez.py
# Purpose:
# Group by element and ion; match within 0.015 Angstrom and 0.02 eV.
# Replace only loggf and lower_state_eV in the base table.
# Preserve the original header order and separator for iSpec compatibility.
# Export matched rows only, sorted by wavelength.
#
# Configure the input and output paths.

import os
import io
import numpy as np
import pandas as pd
from pathlib import Path

# ---------------- paths ----------------
mel_path  = "../input/linelists/linelist_melendez2014/linelist_melendez2014_ispec.tsv"
base_path = "../input/linelists/transitions/VALD.300_1100nm/atomic_lines.tsv"
out_dir   = Path("../input/linelists/transitions/Melendez2014")
out_dir.mkdir(parents=True, exist_ok=True)
out_lines = out_dir / "atomic_lines.tsv"              # Successfully crossmatched rows only.
out_log   = out_dir / "crossmatch_report.csv"         # Optional match report.

# ---------------- tolerances ----------------
TOL_WAVE_A = 0.015   # Å
TOL_ELOW   = 0.020   # eV

# ---------------- helpers ----------------
def to_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def parse_symbol_and_ion_from_element_str(s):
    """ 'Fe 1' / 'Ag 2' -> ('Fe',1) """
    if not isinstance(s, str):
        return "", None
    parts = s.strip().split()
    if not parts:
        return "", None
    sym = parts[0]
    ion = None
    if len(parts) > 1 and parts[1].isdigit():
        ion = int(parts[1])
    return sym, ion

def read_header_raw(filepath):
    """Read the original header order directly, without pandas."""
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        first = f.readline().strip("\n\r")
    # Accept tabs or commas; iSpec normally uses tabs.
    if "\t" in first:
        sep = "\t"
        cols = [c.strip() for c in first.split("\t")]
    else:
        sep = ","
        cols = [c.strip() for c in first.split(",")]
    return cols, sep

# ---------------- read Meléndez ----------------
mel = pd.read_csv(mel_path, sep="\t", dtype=str, keep_default_na=False)

req_mel = ["element", "ion", "wave_A", "lower_state_eV", "loggf"]
missing = [c for c in req_mel if c not in mel.columns]
if missing:
    raise RuntimeError(f"Melendez table is missing required columns: {missing}")

mel["_sym_from_element"], mel["_ion_from_element"] = zip(*mel["element"].map(parse_symbol_and_ion_from_element_str))
mel["_sym"] = np.where(mel["_sym_from_element"].astype(str).str.len()>0,
                       mel["_sym_from_element"],
                       mel["element"].astype(str).str.strip())
mel["_ion"]    = mel["ion"].apply(lambda v: int(v) if str(v).strip().isdigit() else np.nan)
mel["_ion"]    = mel["_ion"].fillna(mel["_ion_from_element"]).astype("Int64")
mel["_wave_A"] = mel["wave_A"].apply(to_float)
mel["_elow"]   = mel["lower_state_eV"].apply(to_float)
mel["_loggf"]  = mel["loggf"].apply(to_float)
mel = mel.dropna(subset=["_sym","_ion","_wave_A","_elow","_loggf"]).copy()

# ---------------- read base & capture its raw header/sep ----------------
base_cols_raw, base_sep = read_header_raw(base_path)   # Preserve the original header order and separator.
base = pd.read_csv(base_path, sep=base_sep, dtype=str, keep_default_na=False)

# Base columns required for matching and replacement.
need_base = ["element","ion","wave_A","lower_state_eV","loggf"]
missing_b = [c for c in need_base if c not in base.columns]
if missing_b:
    raise RuntimeError(f"Base table is missing required columns: {missing_b}\nAvailable columns: {list(base.columns)}")

# Parse element, ion, and numeric fields in the base table.
base["_sym_from_element"], base["_ion_from_element"] = zip(*base["element"].map(parse_symbol_and_ion_from_element_str))
base["_sym"] = np.where(base["_sym_from_element"].astype(str).str.len()>0,
                        base["_sym_from_element"],
                        base["element"].astype(str).str.strip())
base["_ion"]    = base["ion"].apply(lambda v: int(v) if str(v).strip().isdigit() else np.nan)
base["_ion"]    = base["_ion"].fillna(base["_ion_from_element"]).astype("Int64")
base["_wave_A"] = base["wave_A"].apply(to_float)
base["_elow"]   = base["lower_state_eV"].apply(to_float)
base["_loggf"]  = base["loggf"].apply(to_float)

# Drop rows with missing keys and retain original indices for updates.
base_valid = base.dropna(subset=["_sym","_ion","_wave_A","_elow"]).copy()
base_valid.reset_index(inplace=True)
base_valid.rename(columns={"index":"_rowid"}, inplace=True)

# ---------------- bucket by (sym, ion) ----------------
mel_g  = mel.groupby(["_sym","_ion"])
base_g = base_valid.groupby(["_sym","_ion"])
keys   = sorted(set(mel_g.groups.keys()) & set(base_g.groups.keys()))

matched_rowids = []      # Matched indices in the original base table.
report_rows    = []

for key in keys:
    sym, ion = key
    ms = mel_g.get_group(key)
    bs = base_g.get_group(key)
    # Search within each element/ion group for every Melendez line.
    for _, mrow in ms.iterrows():
        mw, me = mrow["_wave_A"], mrow["_elow"]
        cand = bs[(np.abs(bs["_wave_A"] - mw) <= TOL_WAVE_A) &
                  (np.abs(bs["_elow"]   - me) <= TOL_ELOW)]
        if cand.empty:
            continue
        # Choose the closest normalized distance.
        dw = (cand["_wave_A"] - mw)/TOL_WAVE_A
        de = (cand["_elow"]   - me)/TOL_ELOW
        score = np.sqrt(dw**2 + de**2)
        picked = cand.loc[score.idxmin()]
        rid = int(picked["_rowid"])

        # Record the match.
        matched_rowids.append(rid)
        report_rows.append({
            "element": f"{sym} {ion}",
            "mel_wave_A": mw, "mel_E_low": me, "mel_loggf": mrow["_loggf"],
            "base_rowid": rid,
            "base_wave_A_before": picked["_wave_A"],
            "base_E_low_before":   picked["_elow"],
            "base_loggf_before":   picked["_loggf"],
            "d_wave_A": float(picked["_wave_A"] - mw),
            "d_E_low":  float(picked["_elow"]   - me)
        })

# Use each base row at most once.
matched_rowids = sorted(set(matched_rowids))

# Build the updated DataFrame using matched rows only.
# Select rows, replace loggf and lower_state_eV, restore column order, and save.
out_df = base.loc[matched_rowids].copy()

# Map element, ion, wavelength, and excitation potential to Melendez values.
# Merge candidate rows and select the matched Melendez values.
mel_small = mel[["_sym","_ion","_wave_A","_elow","_loggf"]].copy()
mel_small.rename(columns={"_elow":"_elow_mel","_loggf":"_loggf_mel"}, inplace=True)

tmp = out_df.copy()
tmp["_sym_from_element"], tmp["_ion_from_element"] = zip(*tmp["element"].map(parse_symbol_and_ion_from_element_str))
tmp["_sym"] = np.where(tmp["_sym_from_element"].astype(str).str.len()>0,
                       tmp["_sym_from_element"], tmp["element"].astype(str).str.strip())
tmp["_ion"]    = tmp["ion"].apply(lambda v: int(v) if str(v).strip().isdigit() else np.nan)
tmp["_ion"]    = tmp["_ion"].fillna(tmp["_ion_from_element"]).astype("Int64")
tmp["_wave_A"] = tmp["wave_A"].apply(to_float)
tmp["_elow"]   = tmp["lower_state_eV"].apply(to_float)

tmp = tmp.merge(
    mel_small,
    left_on = ["_sym","_ion","_wave_A","_elow"],
    right_on= ["_sym","_ion","_wave_A","_elow_mel"],
    how="left",
    suffixes=("","")
)

# Prefer matched Melendez values; otherwise retain the original values.
# Write text fields without converting missing values to the string nan.
def ffmt(x):  # Use consistent numeric formatting without scientific notation.
    if pd.isna(x):
        return ""
    return f"{float(x):.6f}"

# lower_state_eV
new_elow = np.where(pd.notna(tmp["_elow_mel"]), tmp["_elow_mel"], tmp["_elow"])
out_df["lower_state_eV"] = [ffmt(v) for v in new_elow]

# loggf
new_lgf  = np.where(pd.notna(tmp["_loggf_mel"]), tmp["_loggf_mel"], to_float(out_df["loggf"]))
out_df["loggf"] = [ffmt(v) for v in new_lgf]

# Sort by the original wavelength column when available, otherwise wave_A.
first_col = base_cols_raw[0]
sort_col  = first_col if first_col in out_df.columns else ("wave_A" if "wave_A" in out_df.columns else None)
if sort_col is not None:
    # Attempt numeric sorting.
    so = pd.to_numeric(out_df[sort_col], errors="coerce")
    out_df = out_df.iloc[np.argsort(so.fillna(1e99).values)]

# Export only the original base columns, without intermediate fields.
missing_for_output = [c for c in base_cols_raw if c not in out_df.columns]
if missing_for_output:
    raise RuntimeError(f"Output is missing original base columns; check table conversion: {missing_for_output}")

out_df = out_df[base_cols_raw].copy()

# Convert missing values to empty strings.
out_df = out_df.replace({np.nan:"", "nan":"", "None":""})

# Write using the original separator and consistent decimal formatting.
out_df.to_csv(out_lines, sep=base_sep, index=False, encoding="utf-8", float_format="%.6f")

# Report the conversion.
pd.DataFrame(report_rows).to_csv(out_log, index=False, encoding="utf-8")

print("======== Cross-match Summary ========")
print(f"Melendez input rows         : {len(mel)}")
print(f"Valid base rows             : {len(base_valid)}")
print(f"Common element/ion groups   : {len(keys)}")
print(f"Matched output rows         : {len(out_df)}")
print(f"Output linelist             : {out_lines}")
print(f"Match report                : {out_log}")